# 🎓 Modelo de Machine Learning — Alerta Mensal de Evasão (Pós Live)
### a instituicao | v3 — MBA e Especialização Live/Presencial (`EJND_POS_GRAD`)

Arquitetura de **alerta mensal** com três modelos, um por marco de dias desde o
ingresso de cada aluno. Como a pós tem entrada contínua, o corte é **relativo**
(`DT_INGRESSO + 30 / 60 / 90 dias`), não uma data fixa do calendário.

| Modelo | Marco | Foco operacional | Sinais principais |
|---|---|---|---|
| **M1** | 30 dias | Onboarding e acolhimento | 1º acesso ao AVA, 1ª parcela, entregas no Dropbox |
| **M2** | 60 dias | Engajamento | Gaps de telemetria, 2ª parcela, trabalhos |
| **M3** | 90 dias | Fechamento do módulo | Notas já consolidadas, frequência, 3ª parcela |

---

## ⚠️ Auditoria de 22/09/2026 — cinco defeitos corrigidos

A primeira versão deste modelo reportava topo 50 de **49 / 48 / 49** e PR-AUC de
**0,79 / 0,80 / 0,83**. Esses números **não eram reais**. Cinco defeitos foram
encontrados, confirmados por medição independente e corrigidos.

### 1. 🔴 Vazamento no bloco acadêmico

A CTE `acad_base` juntava `ERP_HISTMATRICULA` **sem nenhum filtro de data**. Duas
contaminações ao mesmo tempo:

- **O futuro.** Notas e frequência vinham do histórico consolidado, ou seja, do
  que aconteceu *depois* do corte. Um aluno com 30 dias de MBA não tem nota
  fechada, mas o modelo lia o boletim do ano seguinte.
- **O passado de outro curso.** O join só por `ALUNO` puxava a vida acadêmica
  inteira da pessoa. Caso real: o aluno `9000004` tem `DT_INGRESSO = 2026-07-24`
  e linhas de **2022**, de um programa anterior.

**A prova:** `NOTA_MEDIA`, `TAXA_APROVACAO`, `FREQ_MEDIA`, `FREQ_MINIMA`,
`DISC_CURSADAS`, `QTD_REPROV_FREQ` e `NOTA_STD` saíam **byte a byte idênticas**
entre o corte de 30 dias e o de 90.

**O sintoma que estava à vista:** os três modelos entregavam o mesmo número
(49,4 / 49,4 / 49,0). Um modelo com 90 dias de dado tem que ser melhor que um
com 30 — e não era, porque a feature dominante era congelada.

**Correção:** `DT_FIM` entre `DT_INGRESSO` e `DT_CORTE_ALUNO − MARGEM_NOTA`.

### 2. 🔴 Censura à direita no rótulo

`EVADIU` vem de `TABELA_ALUNOS.COL_SITUACAO`, o status **atual** do aluno. Não há período
alvo, então o horizonte era diferente para cada um: quem entrou em ago/2025 foi
observado por 13 meses, quem entrou neste mês por zero.

| Tempo desde o ingresso | Alunos | Taxa de evasão |
|---|---|---|
| 0–3 meses | 119 | **10,1%** |
| 3–6 meses | 96 | 25,0% |
| 6–9 meses | 203 | 26,1% |
| 9–12 meses | 85 | 37,6% |
| 12+ meses | 34 | 35,3% |

Monotônico — é propriedade do calendário, não do aluno. É o mesmo defeito que
envenenou a base v1 (ver `CLAUDE.md`).

**Correção:** `MESES_MIN_OBS = 3`. A taxa estabiliza a partir dos 3 meses
(25,0% → 26,1%), então essa é a âncora que remove o grupo com rótulo quebrado ao
menor custo. Medidas também 6 e 9 meses: custam 37% e 77% da base sem comprar
acurácia.

> **Censura residual:** de 9 meses em diante a taxa sobe de novo (37,6%), ou seja,
> há uma segunda onda de desistências que a âncora de 3 meses não captura. Zerar
> isso exigiria âncora de 12 meses, o que deixaria **51 alunos**. Não dá — o
> programa só tem log de AVA desde agosto de 2025.

### 3. 🟠 Data de corte no futuro

`pop` não tinha limite superior em `DT_INGRESSO`. Quem ingressou ontem tinha
corte de 90 dias caindo depois de hoje, com as janelas truncadas: parecia
inativo e estava rotulado como não-evasor.

Medido antes: **22 alunos (4,0%)** no corte de 30 dias, **74 (13,6%)** no de 60,
**108 (19,8%)** no de 90.

### 4. 🟠 `TAXA_APROVACAO` não era uma taxa

O numerador somava **linhas** e o denominador contava **`ORDEM` distinta**.
Medido: mínimo 0, **máximo 12,0**, média 3,50, com 215 dos 545 alunos acima de
1,0. Na prática era "quantas disciplinas o aluno aprovou", ou seja, quão longe
ele chegou — informação do desfecho disfarçada de feature.

### 5. 🟠 A auditoria de viés media alerta, não recall

O numerador era `pred.sum()`, **todos os alertas**, incluindo falsos positivos.
O notebook chegava a reportar **"recall de 177,5% nas mulheres"**. Corrigido para
contar só o acerto (`pred == 1` **e** `EVADIU == 1`).

### Menores

- `FREQ_MEDIA` e `FREQ_MINIMA` eram imputadas com **0,95** — quem não tinha
  registro recebia presença ótima, o oposto do sinal. Agora ficam nulas, como
  manda o `CLAUDE.md` (o nulo é informação e o RandomForest trata NaN).
- `BASE_DIR` estava fixo em `C:/Users/autor/...`. Agora é relativo.
- O `assert` do tópico 2 testava `CONCLUIU == 1`, que é 0 para toda a base —
  passava sempre sem verificar nada. Trocado por três checagens que podem falhar.

---

## O efeito das correções

| | Antes | Depois |
|---|---|---|
| Alunos | 537 | **435** |
| Evasores | 133 (24,8%) | **122 (28,0%)** |
| Topo 50 no M1 | 49,4 | **23,8** |
| Topo 50 no M2 | 49,4 | **28,2** |
| Topo 50 no M3 | 49,0 | **37,4** |
| PR-AUC no M3 | 0,830 | **0,660** |

*(CV out-of-fold, 5 dobras, 5 sementes)*

Os números caíram quase pela metade **porque a metade que sumiu era vazamento**.

O que ganhou sentido foi a progressão: antes os três cortes davam o mesmo número,
agora **23,8 → 28,2 → 37,4**. O modelo de 90 dias é de fato melhor que o de 30, e
é isso que justifica o desenho mensal existir.

---

## ⚠️ Limites desta base

**435 alunos e 122 evasores.** Com 5 dobras são ~24 evasores por dobra. Qualquer
métrica aqui tem margem larga — sempre reporte múltiplas sementes com
desvio-padrão, nunca um número só.

A fila de 50 equivale a **11,5% da base**. Por isso o `recall@10%` é reportado
junto: ele é comparável entre bases de tamanhos diferentes, `recall@K` não é.


In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    recall_score,
    precision_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Caminho relativo: o notebook roda de ml/pos-live-mensal/.
BASE_DIR = os.path.abspath(os.path.join("..", ".."))

# ==============================================================================
# 🎯 PARÂMETROS OPERACIONAIS DO PROJETO (ALTERE AQUI PARA TESTES MANUAIS)
# ==============================================================================
LIMIAR = 0.30       # <-- Altere aqui para testar outros limiares (ex: 0.30 ou 0.25)
FILA = 50           # Capacidade operacional da fila prioritária por ciclo (Top 50)
SEMENTE = 42

CORTES = [(1, "30 dias"), (2, "60 dias"), (3, "90 dias")]
print(f"✅ Configuração ativa: Limiar Operacional = {LIMIAR} | Fila Prioritária = Top {FILA}")

## 1. Carregamento dos Dados
Carregamos as três bases extraídas de `tabelas/`, correspondentes aos cortes temporais M1, M2 e M3.

In [ ]:
DTYPE = {"ALUNO": str, "CURSO": str, "END_MUNICIPIO": str}

def carregar(mes):
    pastas = [
        "../../tabelas",
        "../tabelas",
        "tabelas",
        "C:/Users/autor/Desktop/machine-learning-evasao/tabelas"
    ]
    for pasta in pastas:
        for ext, ler in ((".csv", lambda p: pd.read_csv(p, encoding="utf-8-sig", dtype=DTYPE)),
                         (".xlsx", lambda p: pd.read_excel(p, dtype=DTYPE))):
            p = f"{pasta}/extracao_pos_live_m{mes}{ext}"
            if os.path.exists(p):
                return ler(p)
    raise FileNotFoundError(f"extracao_pos_live_m{mes} nao encontrado.")

bases = {}
linhas = []
for mes, data in CORTES:
    d = carregar(mes)
    bases[mes] = d
    linhas.append({
        "Mês": f"M{mes}",
        "Janela": data,
        "Total Alunos": len(d),
        "Evasores Reais": int(d["EVADIU"].sum()),
        "Taxa de Evasão": f"{d['EVADIU'].mean():.2%}",
        "Alunos c/ AVA Ativo": int(d["AVA_UTILIZOU"].sum()),
        "% AVA": f"{d['AVA_UTILIZOU'].mean():.1%}",
        "Alunos c/ Trabalhos AVA": int((d["AVA_QTD_TRABALHOS"] > 0).sum())
    })

resumo_df = pd.DataFrame(linhas)
print("Resumo dos Datasets Extraídos:")
display(resumo_df)

## 2. Elegibilidade e integridade dos rótulos

`ELEGIVEL == 1` descarta apenas `Transferido`. As três asserções abaixo são
guardas das correções de 22/09/2026 — cada uma falha se um defeito voltar:

| Checagem | Defeito que ela guarda |
|---|---|
| nenhum concluinte com `EVADIU = 1` | rótulo |
| `DT_CORTE_ALUNO <= DT_EXTRACAO` | corte no futuro |
| `TAXA_APROVACAO` dentro de `[0,1]` | a taxa que estourava até 12,0 |

O `assert` anterior testava `CONCLUIU == 1`. Como `COL_SITUACAO` nunca vale
`'Concluido'` nesta base, ele passava sempre sem verificar nada.

A coluna **"com nota fechada até o corte"** mostra o vazamento consertado: 8 no
M1, 120 no M2, 191 no M3. Antes eram os 435 nos três, porque a query lia o
histórico inteiro.


In [ ]:
elegiveis = {}
for mes, data in CORTES:
    d = bases[mes]
    elegiveis[mes] = d[d["ELEGIVEL"] == 1].copy()
    e = elegiveis[mes]

    # O assert antigo testava CONCLUIU == 1, e CONCLUIU e 0 para toda a base
    # (COL_SITUACAO nunca vale 'Concluido' aqui) -- passava sempre sem verificar
    # nada. Estas tres checagens sao as que de fato podem quebrar, e cada uma
    # guarda uma correcao de 22/09/2026:
    falsos = int(e[e["CONCLUIU"] == 1]["EVADIU"].sum())
    assert falsos == 0, f"Mes {mes}: {falsos} concluintes rotulados como evasores"

    futuro = int((pd.to_datetime(e["DT_CORTE_ALUNO"])
                  > pd.to_datetime(e["DT_EXTRACAO"])).sum())
    assert futuro == 0, f"Mes {mes}: {futuro} alunos com corte no futuro"

    t = e["TAXA_APROVACAO"].dropna()
    assert t.empty or t.between(0, 1).all(), (
        f"Mes {mes}: TAXA_APROVACAO fora de [0,1] -- max {t.max()}")

print("Auditoria de elegibilidade e integridade dos rotulos:\n")
for mes, data in CORTES:
    e = elegiveis[mes]
    print(f"   Mes {mes} ({data}): {len(e)} alunos | "
          f"{int(e['EVADIU'].sum())} evasores ({e['EVADIU'].mean():.1%}) | "
          f"{int(e['NOTA_MEDIA'].notna().sum())} com nota fechada ate o corte")

print("\nOK: nenhum concluinte rotulado como evasor, nenhum corte no futuro,")
print("TAXA_APROVACAO dentro de [0,1] nos tres cortes.")


## 3. Engenharia de Features de Risco Combinado
Criamos features com alta sensibilidade combinando comportamento no AVA:
- `RATIO_GAP_ATIVOS`: Proporção de dias sumido sobre dias ativos no AVA.
- `SESSOES_POR_DIA`: Frequência diária de logins na plataforma.

In [ ]:
def engenharia(d):
    d = d.copy()
    d["RATIO_GAP_ATIVOS"] = d["AVA_MAIOR_GAP_DIAS"] / (d["AVA_DIAS_ATIVOS"] + 1)
    d["SESSOES_POR_DIA"] = d["AVA_QTD_SESSOES"] / (d["AVA_DIAS_ATIVOS"] + 1)

    # CURSO e codigo, nao numero -- vira categoria com a cauda longa agrupada.
    # Sao 28 cursos em 428 alunos; sem agrupar, a maioria vira coluna de 2 ou 3
    # linhas e so adiciona ruido.
    vc = d["CURSO"].value_counts()
    d["CURSO_AGRUPADO"] = d["CURSO"].where(d["CURSO"].isin(vc[vc >= 15].index), "OUTRO")

    vm = d["END_MUNICIPIO"].value_counts()
    d["MUNICIPIO_AGRUPADO"] = d["END_MUNICIPIO"].where(
        d["END_MUNICIPIO"].isin(vm[vm >= 15].index), "OUTRO")
    return d


feat = {}
for mes, _ in CORTES:
    feat[mes] = engenharia(elegiveis[mes])

print("Engenharia aplicada aos 3 recortes.\n")
for mes, data in CORTES:
    n_curso = feat[mes]["CURSO_AGRUPADO"].nunique()
    n_mun = feat[mes]["MUNICIPIO_AGRUPADO"].nunique()
    print(f"   M{mes} ({data}): {n_curso} cursos mantidos, {n_mun} municipios "
          f"(cauda longa em 'OUTRO')")


## 4. Separação de $X$ (Features) e $y$ (Target) & Corte Cirúrgico de Ruído
Após diagnósticos estatísticos de Gini e testes de ablator, definimos o vetor final com as **23 variáveis preditivas de maior impacto**, eliminando ruídos nulos e redundâncias:
- **Frequência & Assiduidade:** `FREQ_MEDIA`, `FREQ_MINIMA`, `QTD_REPROV_FREQ`
- **Desempenho Pedagógico:** `TAXA_APROVACAO`, `NOTA_MEDIA`, `NOTA_STD`, `DISC_CURSADAS`
- **Telemetria AVA & Trabalhos:** `AVA_DIAS_ATIVOS`, `AVA_QTD_SESSOES`, `AVA_DIAS_DESDE_ULT_ACESSO`, `AVA_DIAS_ATE_1O_ACESSO`, `AVA_MAIOR_GAP_DIAS`, `RATIO_GAP_ATIVOS`, `SESSOES_POR_DIA`, `AVA_PCT_NOTURNO`, `AVA_PCT_FIM_DE_SEMANA`, `AVA_QTD_TRABALHOS`
- **Histórico Financeiro:** `FIN_VALOR_MEDIO_PARCELA`, `FIN_QTD_EM_ABERTO`, `FIN_MAX_DIAS_ATRASO_PAGO`, `FIN_QTD_TITULOS`
- **Perfil & Bolsa:** `BOLSA_PERC`, `IDADE_INGRESSO`

In [ ]:
# ==================================================================
# AS FEATURES DEVOLVIDAS -- medido em 22/09/2026
#
# A versao anterior do notebook dizia ter "removido 22 variaveis inuteis" apos
# testes de importancia. Aquele corte foi decidido com o modelo VAZADO, quando
# o bloco academico lia o futuro e dominava tudo -- contra uma feature que le a
# resposta, qualquer outra parece inutil. A decisao valia zero e foi refeita.
#
# Ablacao (CV out-of-fold, 5 dobras, 5 sementes, evasores no topo 50):
#
#   bloco devolvido                     ganho 30d   ganho 60d   ganho 90d
#   + AVA recentes                          0,0        +2,6        +0,2
#   + financeiro detalhado                 +0,6        +0,6        +0,8
#   + CURSO agrupado                       +1,6        +3,8        -0,2
#   + perfil (periodo, municipio)          -1,6        -2,2         0,0
#   TODOS JUNTOS                           +3,6        +5,4        +0,8
#
# Resultado: 23,8 -> 27,4 (30d) | 28,0 -> 33,4 (60d) | 37,8 -> 38,6 (90d).
# O acerto por ligacao do M2 sobe de 56% para 67%.
#
# CURSO e o maior ganho isolado e esta disponivel no DIA DA MATRICULA:
# correlacao de +0,246 com a evasao, mais forte que qualquer feature de AVA no
# corte de 30 dias. Alguns MBAs evadem muito mais que outros.
#
# RESSALVA: PER_INGRESSO e MUNICIPIO_AGRUPADO medem NEGATIVO sozinhos (-1,6 e
# -2,2) e so entram porque a combinacao completa mediu melhor -- diferenca de
# 0,4 evasor, dentro do desvio-padrao. Tirar os dois custa quase nada e deixa o
# modelo mais enxuto; e uma escolha legitima.
#
# VIES: CURSO e informante de genero (achado do modelo hibrido). Medido aqui, o
# gap nao piora de forma consistente: 15,6 -> 13,0 pp (30d), 6,2 -> 5,8 (60d),
# 13,8 -> 14,9 (90d). Com 122 evasores divididos por sexo a margem e larga --
# ver a auditoria do topico 9.
# ==================================================================
FEATURES_BASE = [
    # 1. Frequencia e presenca oficial
    "FREQ_MEDIA", "FREQ_MINIMA", "QTD_REPROV_FREQ",

    # 2. Desempenho curricular
    "TAXA_APROVACAO", "NOTA_MEDIA", "NOTA_STD", "DISC_CURSADAS",

    # 3. Telemetria do AVA -- nivel
    "AVA_DIAS_ATIVOS", "AVA_QTD_SESSOES", "AVA_DIAS_DESDE_ULT_ACESSO",
    "AVA_DIAS_ATE_1O_ACESSO", "AVA_MAIOR_GAP_DIAS", "RATIO_GAP_ATIVOS",
    "SESSOES_POR_DIA", "AVA_PCT_NOTURNO", "AVA_PCT_FIM_DE_SEMANA",
    "AVA_QTD_TRABALHOS",

    # 3b. Telemetria do AVA -- janela recente (DEVOLVIDO)
    # No corte de 90 dias, AVA_SESSOES_RECENTES e AVA_DIAS_ATIVOS_RECENTES sao
    # a 1a e a 2a features mais fortes do modelo (12,0% e 11,0%).
    "AVA_SESSOES_RECENTES", "AVA_DIAS_ATIVOS_RECENTES", "AVA_TAXA_DIAS_ATIVOS",
    "AVA_UTILIZOU",

    # 4. Financeiro -- agregados
    "FIN_VALOR_MEDIO_PARCELA", "FIN_QTD_EM_ABERTO", "FIN_MAX_DIAS_ATRASO_PAGO",
    "FIN_QTD_TITULOS",

    # 4b. Financeiro -- comportamento de pagamento (DEVOLVIDO)
    "FIN_QTD_PAGO_ATRASO", "FIN_QTD_PAGO_EM_DIA", "FIN_PROP_PAGO_ANTECIPADO",
    "FIN_MEDIA_DIAS_ATRASO",

    # 5. Perfil e contexto (DEVOLVIDO)
    "BOLSA_PERC", "IDADE_INGRESSO", "PER_INGRESSO",
    "CURSO_AGRUPADO", "MUNICIPIO_AGRUPADO",
]

# ==================================================================
# CORTE DE COLUNAS MORTAS -- por marco
#
# Depois de travar o bloco academico em DT_FIM (topico 2), varias colunas ficam
# vazias nos marcos precoces -- e isso esta CERTO: aos 30 dias de MBA quase
# ninguem fechou disciplina. Mas coluna 100% nula ou com um unico valor nao
# ensina nada, e e ela que abre os buracos brancos na matriz de correlacao
# (correlacao com constante e indefinida: divide por desvio-padrao zero).
#
# Medido: desempenho igual, mas a variancia do M1 cai pela metade
# (+-2,6 -> +-1,7 evasores no topo 50), o que importa numa base de 428.
# ==================================================================
def cortar_mortas(d, cols, rotulo):
    """Tira coluna quase toda nula ou com um unico valor NESTE marco."""
    mortas = [c for c in cols
              if d[c].isna().mean() > 0.90 or d[c].nunique(dropna=True) <= 1]
    if mortas:
        print(f"   {rotulo}: {len(mortas)} mortas cortadas -> {mortas}")
    return [c for c in cols if c not in mortas]


FEATURES_POR_CORTE, X, y, df_audit = {}, {}, {}, {}
print("Colunas mortas por marco:\n")
for mes, data in CORTES:
    d = feat[mes]
    FEATURES_POR_CORTE[mes] = cortar_mortas(d, FEATURES_BASE, f"M{mes} ({data})")
    y[mes] = d["EVADIU"].astype(int).values
    df_audit[mes] = d[["ALUNO", "SEXO", "EVADIU"]].copy()
    X[mes] = d[FEATURES_POR_CORTE[mes]].copy()

print()
for mes, data in CORTES:
    print(f"   M{mes} ({data}): {len(FEATURES_POR_CORTE[mes])} features ativas "
          f"de {len(FEATURES_BASE)}")


## 5. Encoding e Imputação pela Mediana
Como todas as features selecionadas são numéricas ou taxas tratadas, aplicamos o alinhamento de colunas e garantimos que a imputação por mediana seja calculada sem vazamento.

In [ ]:
# CURSO_AGRUPADO e MUNICIPIO_AGRUPADO sao categoricas -- viram one-hot aqui.
# A imputacao NAO acontece nesta celula de proposito: ela e feita com a mediana
# do TREINO de cada dobra, dentro da validacao cruzada (topico 6). Calcular a
# mediana sobre a base inteira vazaria informacao do conjunto de avaliacao.
X_enc_full = {}
for mes, _ in CORTES:
    cat = X[mes].select_dtypes(include=["object", "category", "string"]).columns.tolist()
    e = pd.get_dummies(X[mes], columns=cat, drop_first=True) if cat else X[mes].copy()
    X_enc_full[mes] = e.drop(columns=[c for c in e.columns if e[c].isna().all()])

# ==================================================================
# ENXUGAMENTO PARA AS 20 MAIS IMPORTANTES -- medido em 22/09/2026
#
# Devolver as features cortadas levou o modelo a 40-47 colunas depois do
# one-hot de curso e municipio. Numa base de 428 alunos isso da 9 LINHAS POR
# COLUNA -- para comparar, o hibrido tem 251 e o presencial mensal 118.
#
# Nao e que a Pos Live tenha mais features que os outros (o presencial usa 77);
# e que ela tem 20x menos alunos para sustenta-las.
#
# Medido (CV out-of-fold, 5 dobras, 5 sementes, evasores no topo 50):
#
#                        colunas   30d          60d          90d
#   tudo (40-47)          40-47   27,4 ±1,9    33,4 ±1,4    38,6 ±1,4
#   sem municipio/periodo 36-43   27,0 ±1,5    33,4 ±1,6    37,8 ±1,5
#   top 25                   25   26,6 ±1,9    34,4 ±0,8    38,2 ±1,0
#   TOP 20                   20   27,6 ±2,1    34,4 ±1,9    38,8 ±1,2
#   top 15                   15   27,0 ±1,4    30,6 ±2,1    38,6 ±1,0
#
# O top 20 empata ou ganha nos tres marcos com menos da metade das colunas, e a
# razao sobe de 9 para 21 alunos por coluna. O top 15 quebra no marco de 60
# dias (33,4 -> 30,6), entao 20 e o ponto.
#
# RESSALVA METODOLOGICA: a importancia e calculada sobre a base inteira e a
# avaliacao e por CV na mesma base, o que torna a estimativa levemente
# otimista. Por isso o corte NAO e defendido como ganho de desempenho -- ele
# empata. E defendido por PARCIMONIA: mesmo resultado com metade das colunas,
# numa base pequena demais para sustentar 47.
# ==================================================================
TOP_N = 20

X_enc, IMPORTANCIA_SELECAO = {}, {}
for mes, data in CORTES:
    Xf = X_enc_full[mes]
    if Xf.shape[1] <= TOP_N:
        X_enc[mes] = Xf.copy()
        continue
    imps = []
    for s in (42, 7, 13, 2024, 99):
        m = RandomForestClassifier(n_estimators=200, max_depth=6,
                                   min_samples_split=5, class_weight="balanced",
                                   random_state=s, n_jobs=-1)
        m.fit(Xf.fillna(Xf.median()), y[mes])
        imps.append(m.feature_importances_)
    imp = pd.Series(np.mean(imps, axis=0), index=Xf.columns).sort_values(ascending=False)
    IMPORTANCIA_SELECAO[mes] = imp
    X_enc[mes] = Xf[imp.head(TOP_N).index]

print(f"Categóricas codificadas: {cat}\n")
for mes, data in CORTES:
    n = X_enc[mes].isna().mean().sort_values(ascending=False)
    com_nulo = int((n > 0).sum())
    pior = f"{n.index[0]} ({100*n.iloc[0]:.0f}%)" if com_nulo else "nenhuma"
    print(f"   M{mes} ({data}): {X_enc_full[mes].shape[1]} colunas após encoding "
          f"-> {X_enc[mes].shape[1]} após o top {TOP_N} | "
          f"{len(y[mes]) / X_enc[mes].shape[1]:.0f} alunos por coluna | "
          f"{com_nulo} com nulo (maior: {pior})")

print("\nAs 8 mais fortes de cada marco:")
for mes, data in CORTES:
    if mes in IMPORTANCIA_SELECAO:
        top = IMPORTANCIA_SELECAO[mes].head(8)
        print(f"\n   M{mes} ({data}):")
        for nome, v in top.items():
            print(f"      {nome:34s} {100*v:5.2f}%")


## 6. Divisão Treino e Teste / Validação Cruzada Estratificada (CV Out-Of-Fold)
Aplicamos **Stratified K-Fold (5 dobras)** para coletar as probabilidades **out-of-fold (`proba_oof`)**. Cada aluno recebe sua pontuação estritamente do modelo que não o viu no treino.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
proba_oof = {}

for mes, _ in CORTES:
    proba_oof[mes] = np.zeros(len(y[mes]))
    X_m = X_enc[mes].copy()
    y_m = y[mes]
    
    for train_idx, val_idx in cv.split(X_m, y_m):
        X_tr, y_tr = X_m.iloc[train_idx].copy(), y_m[train_idx]
        X_va = X_m.iloc[val_idx].copy()
        
        med = X_tr.median()
        X_tr = X_tr.fillna(med)
        X_va = X_va.fillna(med)
        
        rf_fold = RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_split=5,
            class_weight="balanced",
            random_state=SEMENTE,
            n_jobs=-1
        )
        rf_fold.fit(X_tr, y_tr)
        proba_oof[mes][val_idx] = rf_fold.predict_proba(X_va)[:, 1]

print("✅ Validação Cruzada Estratificada (5 folds) executada para M1, M2 e M3.")

## 7. Treinamento dos Modelos Finais em Produção
Treinamos os três modelos consolidados de Random Forest sobre a totalidade de cada base mensal.

In [ ]:
modelos_finais = {}
medianas_treino = {}

for mes, data in CORTES:
    X_m = X_enc[mes].copy()
    med = X_m.median()
    medianas_treino[mes] = med
    X_m_imp = X_m.fillna(med)
    
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_split=5,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=-1
    )
    rf.fit(X_m_imp, y[mes])
    modelos_finais[mes] = rf
    print(f"🌲 Modelo Mês {mes} ({data}) treinado: 200 árvores, {X_m.shape[1]} colunas limpas, peso balanceado.")

## 8. Otimização do Limiar de Decisão (*Threshold Tuning* de 0,20 a 0,50)
Avaliamos a sensibilidade (Recall) e o volume de alarmes gerados conforme o limiar varia de 0,20 a 0,50.

In [ ]:
linhas_thresh = []
for mes, data in CORTES:
    for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
        pred = (proba_oof[mes] >= t).astype(int)
        linhas_thresh.append({
            "Mês": f"M{mes}",
            "Janela": data,
            "Limiar": f"{t:.2f}",
            "Recall": f"{recall_score(y[mes], pred, zero_division=0):.1%}",
            "Precisão": f"{precision_score(y[mes], pred, zero_division=0):.1%}",
            "F1-Score": f"{f1_score(y[mes], pred, zero_division=0):.3f}",
            "Alarmes": int(pred.sum()),
            "% Base": f"{pred.sum() / len(y[mes]):.1%}"
        })

df_thresh = pd.DataFrame(linhas_thresh)
display(df_thresh[df_thresh["Limiar"].isin(["0.50", "0.35", "0.30", "0.25", "0.20"])])

## 9. Avaliação no Limiar Operacional e Auditoria Ética de Viés

O limiar está travado em **0,30**, o mesmo dos demais modelos do projeto. Mas
como a seção anterior mostra, ele marca 79% da base no marco de 30 dias — não é
ele que decide quem recebe contato.

Por isso a tabela traz as duas leituras lado a lado, e a que manda é a segunda:

| Métrica | O que responde |
|---|---|
| `Recall @0,30` e `Alarmes` | quantos evasores estão *em algum lugar* da lista, e o tamanho dela |
| **`Topo 50` e `Precisão@50`** | **quantos a equipe realmente alcança ligando para 50** |

`recall@10%` aparece junto porque é comparável entre bases de tamanhos
diferentes — `recall@K` sozinho não é (regra do `CLAUDE.md`).

### Auditoria de viés

`SEXO` não entra no `X`, por decisão ética do `CLAUDE.md`, mas o desvio pode
entrar por proxy. A fórmula conta **acerto** (`pred == 1` **e** `EVADIU == 1`) —
até 22/09/2026 ela contava alerta, e chegava a reportar "recall de 177,5%".

> Leia o gap junto da **taxa de evasão real** de cada grupo, que não é igual aqui.
> Quando um grupo tem risco maior de verdade, o modelo o pontua mais alto e o
> recall sobe junto — parte do desvio é o modelo refletindo a realidade, não
> inventando desigualdade. Ver a ressalva 5 do relatório.


In [ ]:
def calcular_metricas(ym, p, audit_df, fila_n=FILA, t_opt=LIMIAR):
    pred = (p >= t_opt).astype(int)
    ordem = np.argsort(-p)
    topo = ym[ordem[:fila_n]]
    
    audit = audit_df.copy()
    audit["pred"] = pred
    audit["prob"] = p
    
    # CORRIGIDO 22/09/2026 -- antes o numerador era pred.sum(), ou seja TODOS
    # os alertas, inclusive falsos positivos. Recall conta so o acerto:
    # pred == 1 E EVADIU == 1. O bug produzia "recall de 177,5% nas mulheres",
    # e o gap reportado nao media nada.
    def recall_grupo(g):
        sel = audit["SEXO"] == g
        reais = (sel & (audit["EVADIU"] == 1)).sum()
        acertos = (sel & (audit["pred"] == 1) & (audit["EVADIU"] == 1)).sum()
        return acertos / max(reais, 1)

    recall_f, recall_m = recall_grupo("F"), recall_grupo("M")
    
    return {
        "Topo 50": int(topo.sum()),
        "Recall@50": f"{topo.sum() / ym.sum():.1%}",
        "Precisão@50": f"{topo.sum() / fila_n:.1%}",
        "Recall@10%": f"{ym[ordem[:int(0.10 * len(ym))]].sum() / ym.sum():.1%}",
        "PR-AUC": f"{average_precision_score(ym, p):.3f}",
        "ROC-AUC": f"{roc_auc_score(ym, p):.3f}",
        f"Recall @{t_opt:.2f}": f"{recall_score(ym, pred):.1%}",
        f"Precisão @{t_opt:.2f}": f"{precision_score(ym, pred):.1%}",
        "Alarmes": int(pred.sum()),
        "Recall Mulheres": f"{recall_f:.1%}",
        "Recall Homens": f"{recall_m:.1%}",
        "Gap de Viés (pp)": f"{abs(recall_f - recall_m) * 100:.1f} pp"
    }

resultados = [
    {"Mês": f"M{mes}", "Janela": data, **calcular_metricas(y[mes], proba_oof[mes], df_audit[mes])}
    for mes, data in CORTES
]
df_resultados = pd.DataFrame(resultados)
print(f"📊 Desempenho Operacional e Auditoria de Viés (Limiar = {LIMIAR}):")
display(df_resultados)

## 10. A Política de Fila: por que não repetir importa mais que o modelo

Com alerta em três marcos, a pergunta deixa de ser "quantos evasores pego neste
marco" e passa a ser **"quantos evasores distintos alcanço nos 90 dias"**.

São os **mesmos alunos** nos três marcos — a fotografia é que muda. Então quem
está em risco aos 30 dias tende a continuar em risco aos 60, e a lista se repete.

Duas políticas, o mesmo número de ligações:

- **Ingênua** — todo marco pega o topo 50 daquele marco.
- **Sem repetir** — todo marco pega os 50 melhores **que ainda não foram
  ligados**.


In [ ]:
por_marco = {mes: pd.DataFrame({"aluno": feat[mes]["ALUNO"].astype(str).values,
                                "p": proba_oof[mes], "evadiu": y[mes]})
             for mes, _ in CORTES}
TOTAL_EVASORES = int(y[3].sum())

resultados_fila = {}
for politica in ("Ingênua", "Sem repetir"):
    ligados, alcancados, hist = set(), set(), []
    for mes, data in CORTES:
        d = por_marco[mes].sort_values("p", ascending=False)
        if politica == "Sem repetir":
            d = d[~d["aluno"].isin(ligados)]
        fila = d.head(FILA)
        repetidos = len(set(fila["aluno"]) & ligados)
        alcancados |= set(fila.loc[fila["evadiu"] == 1, "aluno"])
        ligados |= set(fila["aluno"])
        hist.append({"Política": politica, "Marco": f"M{mes}", "Janela": data,
                     "Evasores na fila": int(fila["evadiu"].sum()),
                     "Já ligados antes": repetidos,
                     "Evasores alcançados": len(alcancados),
                     "% da evasão": f"{len(alcancados)/TOTAL_EVASORES:.1%}"})
    resultados_fila[politica] = (hist, len(ligados), len(alcancados))

display(pd.DataFrame([h for p in resultados_fila for h in resultados_fila[p][0]]))

print(f"\nAo fim dos 3 marcos, com {len(CORTES)*FILA:,} ligações em ambos os casos:")
for p, (_, n_lig, n_alc) in resultados_fila.items():
    print(f"  {p:12s}: {n_lig:4,} alunos distintos | "
          f"{n_alc:3d} evasores alcançados de {TOTAL_EVASORES} "
          f"({n_alc/TOTAL_EVASORES:.1%})")


## 11. Curvas: captura acumulada e Precision-Recall por marco

À esquerda, o que a política de fila entrega ao longo dos 90 dias. À direita, a
qualidade do ranking de cada marco — a linha vermelha é a taxa de evasão da base,
que é o que um modelo aleatório atingiria.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# a) captura acumulada nas duas politicas
for pol, cor in [("Ingênua", "#A0AEC0"), ("Sem repetir", "#1A365D")]:
    h = resultados_fila[pol][0]
    axes[0].plot([int(x["Marco"][1:]) for x in h],
                 [x["Evasores alcançados"] for x in h],
                 "o-", lw=2.6, ms=7, color=cor, label=pol)
axes[0].set_title("Evasores alcançados ao longo dos 90 dias\n"
                  f"({FILA} ligações por marco nos dois casos)", fontweight="bold")
axes[0].set_xlabel("Marco (30, 60, 90 dias)", fontweight="bold")
axes[0].set_ylabel("Evasores distintos alcançados", fontweight="bold")
axes[0].set_xticks([m for m, _ in CORTES])
axes[0].set_xticklabels([f"M{m}\n({d})" for m, d in CORTES])
axes[0].legend()
axes[0].grid(alpha=0.3)

# b) precision-recall por marco
cores = plt.cm.viridis(np.linspace(0.15, 0.85, len(CORTES)))
for (mes, data), cor in zip(CORTES, cores):
    pr, rc, _ = precision_recall_curve(y[mes], proba_oof[mes])
    axes[1].plot(rc, pr, lw=2.2, color=cor,
                 label=f"M{mes} ({data}) — PR-AUC "
                       f"{average_precision_score(y[mes], proba_oof[mes]):.3f}")
axes[1].axhline(y[3].mean(), color="crimson", ls="--", lw=1.4,
                label=f"Baseline ({y[3].mean():.1%})")
axes[1].set_title("Curva Precision-Recall por marco", fontweight="bold")
axes[1].set_xlabel("Recall", fontweight="bold")
axes[1].set_ylabel("Precisão", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 12. Importância das Features — como o sinal muda ao longo do curso

O modelo **muda de natureza** entre os marcos, e é isso que este gráfico mostra.

Aos 30 dias ele prevê pelo **perfil de entrada** — bolsa, idade, valor da parcela
— porque o aluno mal começou e não gerou comportamento. Aos 90 dias é
**comportamento puro**: as features de acesso ao AVA dominam.

É a justificativa de existirem três modelos em vez de um.


In [ ]:
imp = pd.DataFrame({f"M{mes}": pd.Series(modelos_finais[mes].feature_importances_,
                                          index=X_enc[mes].columns)
                    for mes, _ in CORTES}).fillna(0)
top = imp["M3"].sort_values(ascending=False).head(12).index

fig, ax = plt.subplots(figsize=(11, 6))
for f in top:
    ax.plot([m for m, _ in CORTES], [100 * imp.loc[f, f"M{m}"] for m, _ in CORTES],
            "o-", lw=2, ms=5, label=f)
ax.set_title("Importância das features ao longo do curso", fontweight="bold", pad=12)
ax.set_xlabel("Marco do corte", fontweight="bold")
ax.set_ylabel("Importância relativa (%)", fontweight="bold")
ax.set_xticks([m for m, _ in CORTES])
ax.set_xticklabels([f"M{m}\n({d})" for m, d in CORTES])
ax.legend(fontsize=8, ncol=2, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 8 de cada marco:\n")
display(pd.DataFrame({f"M{m} ({d})": imp[f"M{m}"].sort_values(ascending=False)
                      .head(8).index.tolist() for m, d in CORTES}))


## 13. Salvamento do Artefato em Pickle
Exportamos o artefato oficial contendo os modelos, features, medianas, probabilidades e parâmetros operacionais.

In [ ]:
PKL_LOCAL = "modelo_evasao_pos_live_mensal.pkl"
PKL_TESTES = os.path.join(BASE_DIR, "area-testes", "pos-live-mensal",
                          "modelo_evasao_pos_live_mensal.pkl")

# ==================================================================
# O QUE VAI NO ARTEFATO -- corrigido em 23/09/2026
#
# A versao anterior gravava em "features" a lista FEATURES_POR_CORTE, que e a
# lista ANTES do one-hot (34 nomes no M3, incluindo CURSO_AGRUPADO e
# MUNICIPIO_AGRUPADO como categoricas cruas). Mas o modelo espera as 20 colunas
# JA CODIFICADAS do top 20. Quem carregasse o .pkl para pontuar turma nova
# montaria X com 34 colunas contra um modelo de 20 -- ou quebra, ou alinha
# errado em silencio.
#
# Os cadernos de area-testes/ nao pegavam isso porque consomem proba_oof, que ja
# vem pronta. Qualquer codigo de producao quebraria.
#
# Agora ha tres chaves, com papeis distintos:
#   colunas_modelo  -- o que o modelo REALMENTE espera (entrada do predict)
#   features_pre_encoding -- a lista antes do one-hot, para auditoria
#   agrupamentos    -- as categorias mantidas, para reproduzir o one-hot igual
# ==================================================================
artefato = {
    "modelos": modelos_finais,
    "colunas_modelo": {m: X_enc[m].columns.tolist() for m, _ in CORTES},
    "features_pre_encoding": FEATURES_POR_CORTE,
    "agrupamentos": {
        m: {"cursos": sorted(feat[m]["CURSO_AGRUPADO"].unique().tolist()),
            "municipios": sorted(feat[m]["MUNICIPIO_AGRUPADO"].unique().tolist())}
        for m, _ in CORTES},
    "medianas": medianas_treino,
    "proba_oof": proba_oof,
    "limiar": LIMIAR,
    "fila": FILA,
    "cortes": CORTES,
    "top_n": TOP_N,
}

# guarda-chuva: o que esta gravado tem que casar com o que o modelo aceita
for m, _ in CORTES:
    n_art = len(artefato["colunas_modelo"][m])
    n_mod = modelos_finais[m].n_features_in_
    assert n_art == n_mod, (
        f"M{m}: artefato guarda {n_art} colunas mas o modelo espera {n_mod}")

joblib.dump(artefato, PKL_LOCAL, compress=3)
joblib.dump(artefato, PKL_TESTES, compress=3)

print("Artefato serializado nos dois locais:")
for p in (PKL_LOCAL, PKL_TESTES):
    print(f"   {os.path.abspath(p)}  "
          f"({os.path.getsize(p)/1024:.0f} KB)")
print("\nColunas por marco (artefato x modelo):")
for m, data in CORTES:
    print(f"   M{m} ({data}): {len(artefato['colunas_modelo'][m])} colunas "
          f"= {modelos_finais[m].n_features_in_} esperadas pelo modelo  OK")


## 14. Matriz de Correlação de Pearson & Diagnóstico de Multicolinearidade
Em conformidade com a Fase 5 do Processo Padrão:
1. **Heatmap Triangular Ordenado** (1.650 x 1.350 px, paleta `coolwarm`, máscara triangular superior e `EVADIU` na 1ª coluna).
   - **Zero dados cinzas / vazios**: Remoção de constantes (`CONCLUIU`, `ELEGIVEL`) e imputação de medianas prévia ao cálculo de Pearson.
2. **Ranking Horizontal $\pm r$** das variáveis mais correlacionadas com evasão (+r risco e -r retenção).
3. **Tabela de Multicolinearidade** diagnosticando pares com $|r| \ge 0,70$.

In [ ]:
# ==================================================================
# MATRIZ DE CORRELACAO -- padrao do projeto
#
# Tres coisas mudaram em 22/09/2026, para ficar igual aos outros modelos:
#
# 1. Construida no ULTIMO corte (M3), nao no primeiro. O M1 tem 7 colunas
#    inertes e a matriz saia cheia de buraco branco.
# 2. Escala de +-0,5 em vez de +-1,0. Correlacao com o alvo raramente passa de
#    0,5; na escala cheia tudo vira rosa claro e nao se le nada.
# 3. SEM imputar mediana so para "sumir com o cinza". A versao anterior fazia
#    isso, e numa coluna 97% nula a mediana a torna 97% constante -- a
#    correlacao vai a zero e o problema fica escondido em vez de aparecer.
# ==================================================================
MES_REF = 3   # o corte com mais dado fechado

cols_corr = X_enc[MES_REF].columns.tolist()
dc = X_enc[MES_REF][cols_corr].copy()
dc.insert(0, "EVADIU", y[MES_REF])
corr = dc.corr(method="pearson")

PNG_DIR = os.path.join(BASE_DIR, "relatorios", "matriz-correlacao")
PNG_PATH = os.path.join(PNG_DIR, "matriz_correlacao_pos_live_mensal.png")

plt.figure(figsize=(11, 9))   # 11x9in @150dpi = 1650x1350 px, padrao do projeto
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), cmap="coolwarm",
            vmax=0.5, vmin=-0.5, center=0, square=True, linewidths=0.5,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            cbar_kws={"shrink": 0.8})
plt.title(f"Matriz de Correlação de Pearson — Pós Live, corte M{MES_REF} (90 dias)\n"
          f"({len(dc):,} alunos, {int(y[MES_REF].sum())} evasores)",
          fontsize=12, fontweight="bold", pad=14)
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
if os.path.isdir(PNG_DIR):
    plt.savefig(PNG_PATH, dpi=150)   # sem bbox_inches: ele recorta e quebra o padrao
    print(f"PNG oficial salvo em: {PNG_PATH}")
plt.show()


In [ ]:
# --- ranking: correlacao com EVADIU no primeiro vs no ultimo marco ---------
# E aqui que a comparacao entre cortes fica legivel -- um heatmap por corte,
# lado a lado, vira ruido visual.
comuns = [c for c in X_enc[1].columns if c in X_enc[3].columns]
evol = {}
for mes, _ in CORTES:
    d = X_enc[mes][comuns].copy()
    d["EVADIU"] = y[mes]
    evol[mes] = d.corr()["EVADIU"].drop("EVADIU")
evol = pd.DataFrame(evol).dropna()

ordem = evol[3].abs().sort_values(ascending=False).head(14).index[::-1]
pos = np.arange(len(ordem))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(pos + 0.20, evol.loc[ordem, 1], 0.38, color="#A0AEC0",
        edgecolor="black", lw=0.4, label="M1 (30 dias)")
ax.barh(pos - 0.20, evol.loc[ordem, 3], 0.38, color="#1A365D",
        edgecolor="black", lw=0.4, label="M3 (90 dias)")
ax.axvline(0, color="gray", ls="--", lw=0.9)
ax.set_yticks(pos); ax.set_yticklabels(ordem, fontsize=8.5)
ax.set_xlabel("Correlação de Pearson com EVADIU   —   negativo retém, positivo arrisca",
              fontsize=10, fontweight="bold")
ax.set_title("Quanto cada variável informa sobre a evasão, aos 30 e aos 90 dias",
             fontsize=12, fontweight="bold", pad=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("=" * 76)
print("AS QUE MAIS GANHAM SINAL ENTRE 30 E 90 DIAS")
print("=" * 76)
evol["ganho"] = evol[3].abs() - evol[1].abs()
display(evol.reindex(evol["ganho"].sort_values(ascending=False).index)
            .head(8)[[1, 3, "ganho"]].round(3)
            .rename(columns={1: "M1 (30d)", 3: "M3 (90d)"}))

pares = [{"Feature 1": a, "Feature 2": b, "r": round(corr.loc[a, b], 3)}
         for i, a in enumerate(corr.columns) for b in corr.columns[i+1:]
         if abs(corr.loc[a, b]) >= 0.70]
print("\n" + "=" * 76)
print("MULTICOLINEARIDADE NO CORTE M3 (|r| >= 0,70)")
print("=" * 76)
print(pd.DataFrame(pares).sort_values("r", key=abs, ascending=False).to_string(index=False)
      if pares else "Nenhum par com multicolinearidade severa.")


## 15. O Teste que Decide: o modelo ganha de uma regra de uma coluna?

Critério herdado da pós EAD e registrado no `CLAUDE.md`:

> *"Só publicar modelo que supere a regra de um critério por margem que sobreviva
> ao intervalo de confiança."*

Na pós EAD esse teste **derrubou o modelo**: ordenar por `AVA_DIAS_ATIVOS`, sem
treino nenhum, acertava exatamente os mesmos evasores que vinte features.

Aqui os três competem sobre **os mesmos 428 alunos**, com a mesma métrica — o
`CLAUDE.md` registra que comparar o modelo out-of-fold com o acaso simulado sobre
outra base já inverteu uma conclusão neste projeto.

### Resultado (topo 50, CV out-of-fold, 5 sementes)

| Marco | Modelo | Regra de 1 coluna | Diferença | Em desvios-padrão |
|---|---|---|---|---|
| M1 (30 dias) | **27,2 ±1,6** | 17 | **+10,2** | +6,4 σ |
| M2 (60 dias) | **33,8 ±0,7** | 30 | **+3,8** | +5,1 σ |
| M3 (90 dias) | **39,4 ±1,2** | 37 | **+2,4** | +2,0 σ |

*(o acaso alcançaria 14,3; a regra é `AVA_DIAS_DESDE_ULT_ACESSO`, maior primeiro —
no M1 a melhor é `AVA_DIAS_ATIVOS`, menor primeiro, também com 20)*

**Os três passam.** Mas com folgas muito diferentes, e o padrão é o inverso do
intuitivo: **o modelo vale mais quando tem menos dado.**

Aos 90 dias uma coluna sozinha pega 37 dos 50. Vinte features ganham 2,4 — dois
desvios-padrão, o limite do critério. Faz sentido: nesse ponto quem vai evadir já
sumiu do AVA, e `dias sem acessar` captura isso quase inteiro.

Aos 30 dias ninguém sumiu ainda. O modelo precisa combinar sinais fracos — bolsa,
idade, valor da parcela — e é aí que ele ganha o salário: 27,2 contra 17.

> **Consequência operacional:** se algum dia for preciso simplificar, o **M3 é o
> primeiro candidato a virar regra**. "Ordene por dias sem acessar o AVA" entrega
> 37 em 50, é auditável e explicável ao aluno. O **M1 não tem substituto** — e é
> justamente o alerta mais valioso, com o curso inteiro pela frente.


In [ ]:
# Replica o teste com o pipeline ja montado acima -- os tres competidores
# disputam sobre os MESMOS alunos, com a MESMA metrica.
from scipy.stats import hypergeom

REGRA_OBVIA = ("AVA_DIAS_DESDE_ULT_ACESSO", +1)   # +1 = maior valor, maior risco


def topo_k(ym, score, k=FILA):
    s = pd.Series(score).fillna(-np.inf).values
    return int(ym[np.argsort(-s)[:k]].sum())


linhas_regra = []
for mes, data in CORTES:
    d, ym = feat[mes], y[mes]
    N, EV = len(ym), int(ym.sum())

    mod = topo_k(ym, proba_oof[mes])

    candidatas = [c for c in d.columns
                  if pd.api.types.is_numeric_dtype(d[c])
                  and c not in ("EVADIU", "CONCLUIU", "ELEGIVEL", "DIAS_CORTE_OBS",
                                "DIAS_OBSERVADOS")
                  and d[c].nunique(dropna=True) > 1]
    res = sorted(((c, sg, topo_k(ym, sg * d[c].astype(float)))
                  for c in candidatas for sg in (+1, -1)), key=lambda r: -r[2])
    obvia = next(r for r in res if r[0] == REGRA_OBVIA[0] and r[1] == REGRA_OBVIA[1])

    linhas_regra.append({
        "Marco": f"M{mes}", "Janela": data,
        "Modelo (topo 50)": mod,
        "Regra 1 coluna": obvia[2],
        "Melhor regra": f"{res[0][2]} [{res[0][0]}]",
        "Acaso": f"{FILA/N*EV:.1f}",
        "Modelo - regra": mod - obvia[2],
    })

display(pd.DataFrame(linhas_regra))

print("\nAs 5 melhores regras de uma coluna, por marco:")
for mes, data in CORTES:
    d, ym = feat[mes], y[mes]
    candidatas = [c for c in d.columns
                  if pd.api.types.is_numeric_dtype(d[c])
                  and c not in ("EVADIU", "CONCLUIU", "ELEGIVEL", "DIAS_CORTE_OBS",
                                "DIAS_OBSERVADOS")
                  and d[c].nunique(dropna=True) > 1]
    res = sorted(((c, sg, topo_k(ym, sg * d[c].astype(float)))
                  for c in candidatas for sg in (+1, -1)), key=lambda r: -r[2])[:5]
    print(f"\n   M{mes} ({data}):")
    for c, sg, t in res:
        print(f"      {t:3d}   ordenar por {c} "
              f"({'maior' if sg > 0 else 'menor'} primeiro)")

print("\nObs.: este caderno usa uma semente (a tabela do markdown acima e a media")
print("de 5). A ordem de grandeza e a conclusao sao as mesmas.")
